# 4.5 RNA-seq: Gene Expression in Type 2 Diabetes

**Omics series, class 5 of 7.** So far we've looked at **DNA**: the variants people are born with. DNA is the same in every cell and doesn't change with disease. **Gene expression** does: it shows which genes are **active**, and how much, at the moment a sample was taken. *All of Us* has measured gene expression (**RNA sequencing**, or RNA-seq) in **whole blood** for about 9,000 participants.

Our questions:

1. **Which genes are more or less active in people with type 2 diabetes?**
2. **Back to *TCF7L2*:** is the gene expressed differently in diabetes, and does rs7903146's **T allele** change how much of it is made in blood?

### By the end of this notebook you'll be able to

1. Explain what RNA-seq measures, and the difference between **counts** and **TPM**
2. Find the *All of Us* RNA-seq files, and use their **quality** measures
3. Build a type 2 diabetes cohort of participants with RNA-seq data
4. Load, filter, and **normalize** a gene expression matrix, and explore it with **PCA**
5. Run a **differential expression** analysis with `limma`, and read a volcano plot and a results table
6. Test whether a genetic variant is linked to a gene's expression (an **eQTL**), and look it up in the *All of Us* eQTL results

| | |
|---|---|
| 📂 **Uses** | the *All of Us* RNA-seq and genomic files (v9) and the Controlled Tier dataset |
| 💾 **Saves** | `rnaseq_t2d_results.csv`: one row per gene |

### 🔎 Analysis at a glance

| | |
|---|---|
| ❓ **Questions** | Which genes are more or less active in the **blood** of people with type 2 diabetes? Does rs7903146 change how much *TCF7L2* is made in blood? |
| 🧪 **The analysis** | **Differential expression**: counts **normalized** (TMM) and log-transformed, then a **linear model** per gene with limma-voom, adjusted for age, sex, and RNA quality. Then an **eQTL** test: *TCF7L2* expression ~ copies of T |
| 🎯 **Why** | Raw counts can't be compared until they're normalized for sequencing depth and composition; expression is a number, so linear models; ~29,000 genes, so an adjusted p-value (FDR); limma borrows information across genes, which makes each test more reliable |
| ✅ **What we conclude** | Which genes (and biological processes) differ, and whether the diabetes variant acts through blood expression (in our test run: a few dozen genes, many from **neutrophils**; rs7903146 barely affects *TCF7L2* in blood) |
| ⚠️ **What it can't tell us** | Cause vs consequence vs treatment (e.g. metformin); what happens in **other tissues** such as the pancreas; whether cells changed or the **mix of cells** changed |
| 🔗 **Connections** | The variant comes from **4.1–4.4**; proteins in **4.6**; RNA and protein together in **4.7**. Combining a GWAS (**4.2**) with eQTLs is called **colocalization**: does the diabetes signal and the expression signal come from the same variant? |

### Class plan

| Part | Minutes |
|---|---|
| 1. Key concepts | 15 |
| 2. Setup and a tour of the RNA-seq files | 15 |
| 3. The cohort | 10 |
| 4. Load the counts | 10 |
| 5. Normalize and explore (PCA) | 20 |
| 6. Differential expression | 25 |
| 7. Back to *TCF7L2* and rs7903146 | 20 |
| Wrap-up | 5 |

**Before you start:**

- This notebook works **on its own**: it builds its own cohort.
- Like 4.1, it's an **R** notebook in `notebooks/omics/`, and loads `../aou_setup.R`.
- 👩‍🏫 **Instructor tip:** run the **Setup** cell once before class. Installing the extra packages takes 5–10 minutes the first time.

### ⏱️ Time and 💰 cost

| | |
|---|---|
| ⏱️ **Class time** | 120 minutes |
| ⏱️ **Run time** | about **10 minutes** to run every cell (plus 5–10 min the first time, to install limma and edgeR, and a few more for the optional enrichment packages). Slowest: loading the counts (a few min) and the cohort query (1–2 min) |
| 💰 **Data cost** | **under $1**: one BigQuery query, plus reading the large count file and part of the eQTL results from the *All of Us* bucket (pennies) |
| 💰 **App cost** | about **$0.27 an hour** while the app is running (the default app: 4 CPUs, 26 GB memory), so about **$0.55** for a 2-hour class. The app's disk costs about **$0.70 a day**, even while the app is stopped |
| 💡 **To spend less** | install the packages before class; don't rerun the loading cell; stop the app during long breaks |
| **Estimated total** | about **$1 to $2** per student for this class |

App prices are the Workbench's estimate for the default app (October 2026); yours are on the **Apps** tab when you create or edit an app. BigQuery and bucket costs are our estimates from test runs. **Stop** the app after each class, and **delete** it when you've finished the series (save anything you need to your bucket first: deleting removes everything on the app's disk).

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data (including expression values for individuals), or counts of 1 to 20. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## Part 1. Key concepts

### From genes to reads

Every cell has the same DNA, but each cell only **uses** some of its genes. To use a gene, the cell copies it into **RNA**. The more active a gene is, the more RNA copies it makes.

**RNA-seq** measures this. In the lab, the RNA from a sample is broken into short pieces and sequenced, producing millions of short **reads**. A computer then works out which gene each read came from.

### Counts

The result is a **count matrix**: one row per **gene**, one column per **person**, and each value is the number of reads from that gene in that person.

| | Person A | Person B | Person C |
|---|---|---|---|
| Gene 1 | 1,520 | 980 | 2,310 |
| Gene 2 | 0 | 3 | 1 |

### Why counts need **normalizing**

Raw counts can't be compared directly:

- **Sequencing depth:** if Person A was sequenced twice as deeply as Person B, *every* gene will have about twice as many reads, even if nothing biological differs.
- **Gene length:** a long gene produces more reads than a short gene with the same activity.
- **Composition:** each person's reads are shared out among all their genes. If a few genes are hugely active in one person, they "use up" reads, and every other gene looks lower, even though nothing changed. **TMM** normalization (*trimmed mean of M-values*, from edgeR) corrects for this: it assumes **most genes don't change** between people, and scales each person so those genes line up.

*All of Us* provides two kinds of values:

| | What it is | Use it for |
|---|---|---|
| **Counts** (`gene_reads`) | raw read counts, whole numbers | **comparing groups** (differential expression): the statistical methods expect counts |
| **TPM** (`gene_tpm`) | *transcripts per million*: adjusted for depth **and** gene length, so each person's values add up to 1,000,000 | plots, and comparing genes within a person |

In this class we use **counts**, and normalize them **ourselves**. That's an important point: *All of Us* has done the **processing** (aligning reads to the genome and counting them, with the RNA-SeQC 2 pipeline), but **not** the normalization. **limma doesn't normalize either**: it fits models to whatever values you give it. So normalizing, and **checking** that it worked, is our job (Part 5).

### Log scale

Expression values are very **skewed**: a few genes have huge counts, most have small ones. We work on the **log₂ scale**, where each step of 1 means "twice as much". A difference of **1** on the log₂ scale is a **2-fold** difference; **−1** is half as much.

### Differential expression

For **each gene**, we compare its average expression in people **with** diabetes to people **without**, while adjusting for **age** and **sex**. That's one regression model per gene: about 15,000 models.

- **log fold change (logFC):** the difference between the groups on the log₂ scale. logFC = 1 means twice as high in people with diabetes; logFC = −0.5 means about 30% lower.
- **Multiple testing:** with 15,000 tests, about 750 would have p < 0.05 **by chance alone**. So we use the **adjusted p-value** (the *false discovery rate*, FDR). Genes with **adjusted p < 0.05** are our findings: we expect fewer than 5% of them to be false positives.

### Whole blood

The *All of Us* RNA-seq samples are **whole blood**: a mix of many cell types (mostly white blood cells). So we're measuring blood's gene activity, not the pancreas or liver, where much of type 2 diabetes happens. Differences can also come from a change in the **mix** of cells, not only from each cell changing.

## Part 2. Setup

This notebook needs three extra packages:

| Package | What it does |
|---|---|
| **data.table** | reads very large text files quickly (`fread()`) |
| **edgeR** | stores count data and normalizes it |
| **limma** | runs the differential expression models |

edgeR and limma come from **Bioconductor**, a collection of R packages for biology, so they're installed with `BiocManager::install()` instead of `install.packages()`. The cell below only installs what's missing.

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions

# Extra packages for this notebook. First time only: this can take 5-10 minutes.
user_lib <- Sys.getenv("R_LIBS_USER")
dir.create(user_lib, recursive = TRUE, showWarnings = FALSE)
.libPaths(c(user_lib, .libPaths()))

for (pkg in c("data.table", "BiocManager")) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg, lib = user_lib, repos = "https://cloud.r-project.org")
  }
}
for (pkg in c("limma", "edgeR")) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    BiocManager::install(pkg, lib = user_lib, update = FALSE, ask = FALSE)
  }
}

suppressPackageStartupMessages(library(edgeR))  # also loads limma

RNASEQ_DIR <- file.path(aou_data_dir("v9"), "multiomics", "rnaseq")
list.files(RNASEQ_DIR)
list.files(file.path(RNASEQ_DIR, "rnaseqc2"))

### A tour of the RNA-seq files

**🔍 Read the output:**

| File or folder | What's in it |
|---|---|
| `manifest.tsv` | one row per sample: its IDs and where its raw sequencing file is stored |
| `rnaseq_metadata.tsv` | information **about** each sample (for example, how and when it was processed) |
| `rnaseqc2/` | expression values from the **RNA-SeQC 2** pipeline: `gene_reads` (**counts**, what we use), `gene_tpm` (TPM), `exon_reads` (counts per exon), and `metrics` (quality checks) |
| `rsem/` | expression from a second pipeline, **RSEM**, including **transcript** (isoform) level values |
| `eqtl/`, `sqtl/` | precomputed results linking **genetic variants** to expression (eQTL) and to splicing (sQTL); we use the eQTL results in Part 7 |

The raw sequencing files (**BAM** files) are listed in the manifest. They're huge, and we don't need them for this class.

## The manifest

`read_tsv()` reads a tab-separated file. `col_types = cols(.default = "c")` reads every column as **text**, so long ID numbers are kept exactly as they are.

We only look at the **column names** and **counts**, not the rows: the rows are participant IDs, which we shouldn't display.

In [ ]:
manifest <- read_tsv(file.path(RNASEQ_DIR, "manifest.tsv"), col_types = cols(.default = "c"))

names(manifest)
nrow(manifest)                   # samples
n_distinct(manifest$research_id) # participants

**🔍 Read the output:** `research_id` is the participant's ID: the same number as `person_id` in the Controlled Tier dataset. `sampleid` is the ID of the sample. The two counts are equal: everyone has **one** sample.

Now the sample metadata. Again, column names only:

In [ ]:
rna_metadata <- read_tsv(file.path(RNASEQ_DIR, "rnaseq_metadata.tsv"), col_types = cols(.default = "c"))

names(rna_metadata)

# The RNA Quality Score: a summary across all samples (no individual values)
summary(as.numeric(rna_metadata$rqs))

**🔍 Read the output:** the metadata describes each sample's **quality**:

| Column | Means |
|---|---|
| `sampleid`, `research_id` | the sample's ID, and the participant's ID |
| `rqs` | **RNA Quality Score**, from 1 to 10: how intact the RNA was. RNA breaks down quickly; higher = better |
| `alignment_rate_pct` | the % of reads that matched the genome |
| `mrna_bases_pct`, `ribosomal_bases_pct` | the % of sequence from genes (what we want) vs from ribosomes (mostly unwanted) |
| `reads_aligned_in_pairs`, `mean_insert_size` | more technical measures of the sequencing |
| `processing_status`, `pipeline_id` | the same for every sample: all were processed the same way |

**💬 Discuss:** RNA quality can change expression values for **technical** reasons: degraded RNA loses some genes faster than others. If people with diabetes had lower-quality samples (for example, collected at different sites), quality could look like a diabetes effect. That's why we'll add `rqs` to the model in Part 6, as a **technical covariate**.

There's no **batch** column (for example, which lab plate a sample was on). In many studies, batch is the biggest technical factor; here we can't adjust for it.

### The count file

The counts are in a **GCT** file, a common format for expression data. It's a compressed (`.gz`) text file with:

- **Line 1:** the format version (`#1.2`)
- **Line 2:** the number of genes and the number of samples
- **Line 3:** the column names: `Name` (the gene's Ensembl ID), `Description` (the gene symbol), then **one column per sample**
- **Line 4 onward:** one row per gene

The full file is far too big to look at, so we read just the first 3 lines with `readLines()`.

In [ ]:
counts_file <- file.path(RNASEQ_DIR, "rnaseqc2", "aou_rnaseq_20260413.gene_reads.gct.gz")

gct_top <- readLines(gzfile(counts_file), n = 3)
gct_top[1:2]   # version, then genes and samples

file_columns <- str_split(gct_top[3], "\t")[[1]]
file_columns[1:2]          # the two gene columns
length(file_columns) - 2   # number of sample columns

**🔍 Read the output:** line 2 shows about **78,000 genes** and **9,000 samples**: roughly 700 million numbers. Loading all of it would need more memory than a standard app has. So in Part 4 we'll load **only the columns for the people in our cohort**.

The sample columns are named with an ID. This cell checks which manifest ID they use, then builds a lookup table from each **column name** to the **person_id**:

In [ ]:
sample_columns <- file_columns[-(1:2)]

c(columns_are_research_id = mean(sample_columns %in% manifest$research_id),
  columns_are_sampleid    = mean(sample_columns %in% manifest$sampleid))

column_id_type <- if (mean(sample_columns %in% manifest$research_id) > 0.9) "research_id" else "sampleid"

column_lookup <- manifest %>%
  transmute(column_name = .data[[column_id_type]], person_id = research_id) %>%
  filter(column_name %in% sample_columns) %>%
  distinct(person_id, .keep_all = TRUE)   # one sample per person

nrow(column_lookup)

**🔍 Read the output:** one of the two values should be **1** (100% of columns matched). `column_lookup` now has one row per participant with RNA-seq.

`.data[[column_id_type]]` means "the column whose name is stored in `column_id_type`". It lets the same code work whichever ID the file uses.

## Part 3. The cohort

The same type 2 diabetes cohort as in 4.1 to 4.4, from the shared query `sql/t2d_cohort.sql`: **cases** have type 2 diabetes (and no type 1); **controls** have no diabetes of any type; everyone is an adult with EHR data.

We run the query, then keep only people **with RNA-seq** (`column_lookup`), cases and controls, aged 18 to 100, with sex at birth *Female* or *Male*. We also add each sample's **RNA quality score** (`rqs`) from the metadata.

`factor(cohort, levels = c("control", "case"))` makes **control** the reference group, so results describe people **with** type 2 diabetes compared with people without.

💰 The query costs cents to under $1, and takes a minute or two.

In [ ]:
cohort <- run_query(str_glue(read_file("../sql/t2d_cohort.sql"), AGE_DATE = "2025-01-01")) %>%
  mutate(person_id = as.character(person_id))

# Everyone with RNA-seq, by cohort group
cohort %>% filter(person_id %in% column_lookup$person_id) %>% count(cohort) %>% hide_small_counts()

rna_cohort <- cohort %>%
  filter(person_id %in% column_lookup$person_id,
         cohort %in% c("case", "control"),
         between(age, 18, 100),
         sex_at_birth %in% c("Female", "Male")) %>%
  left_join(rna_metadata %>% transmute(person_id = research_id, rqs = as.numeric(rqs)),
            by = "person_id") %>%
  mutate(group = factor(cohort, levels = c("control", "case")))

rna_cohort %>%
  group_by(group) %>%
  summarise(n          = n(),
            mean_age   = round(mean(age), 1),
            pct_female = round(100 * mean(sex_at_birth == "Female"), 1),
            mean_rqs   = round(mean(rqs, na.rm = TRUE), 2))

**🔍 Read the output:**

- The first table: how many people with RNA-seq are cases, controls, or excluded (e.g. type 1 diabetes).
- The second: our groups. Cases are usually **older**, and the share of women may differ: that's why age and sex go in the model.
- Is the **RNA quality** similar in both groups? If it differs, adjusting for it matters even more.

In our test run: about 1,200 cases and 5,300 controls with RNA-seq; cases were **12 years older** on average (64 vs 52) and less often female (48% vs 55%); RNA quality was almost identical (7.7 vs 7.8).

## Choose the samples for this class


We'll take a **random sample of 300 people from each group**. This keeps the data small enough for a standard app and fast enough for class. With more time or a bigger app, you can increase `N_PER_GROUP` (see *Going further*).

- `arrange(person_id)` then `set.seed()` makes the random sample **repeatable**: everyone who runs this gets the same people (BigQuery doesn't return rows in a fixed order, so we sort first).
- `slice_sample(n = …)` picks that many rows at random **within each group** (because of `group_by()`). If a group has fewer people, it keeps all of them.

In [ ]:
N_PER_GROUP <- 300

set.seed(2026)
analysis_samples <- rna_cohort %>%
  arrange(person_id) %>%
  group_by(group) %>%
  slice_sample(n = N_PER_GROUP) %>%
  ungroup()

analysis_samples %>% count(group)

## Part 4. Load the counts

To read **only our cohort's columns** from a 9,000-column file, we use two command-line tools that run before R sees the data:

| Tool | What it does here |
|---|---|
| `gzip -dc` | uncompresses the `.gz` file as it's read |
| `tail -n +3` | skips the first 2 lines (version and size), so the header is the first line |
| `cut -f 1,2,…` | keeps only the listed **column numbers**: the two gene columns and our samples |

`fread(cmd = …)` from **data.table** runs that command and reads the result. The pipe `|` passes the output of one tool to the next.

⏱️ This takes a few minutes: the whole file still has to be uncompressed once.

In [ ]:
keep <- column_lookup %>% filter(person_id %in% analysis_samples$person_id)
column_numbers <- match(keep$column_name, file_columns)   # positions in the file

read_command <- str_glue(
  "gzip -dc {shQuote(path.expand(counts_file))} | tail -n +3 | cut -f 1,2,{str_c(sort(column_numbers), collapse = ',')}"
)

counts_raw <- data.table::fread(cmd = read_command, sep = "\t", header = TRUE)
dim(counts_raw)

**🔍 Read the output:** about 78,000 rows (genes) and 600 + 2 columns.

Next we split it into:

- `genes`: the gene ID and symbol for each row
- `counts`: a **matrix** of numbers (genes × people). Matrices are what edgeR and limma work with.

Then we rename the columns to `person_id`, and put the sample table in the **same order as the columns**. `stopifnot()` stops with an error if the order doesn't match: a mismatch would silently give every person someone else's data.

In [ ]:
counts_raw <- as.data.frame(counts_raw)

genes  <- counts_raw[, c("Name", "Description")]
counts <- as.matrix(counts_raw[, -(1:2)])
rownames(counts) <- genes$Name
colnames(counts) <- keep$person_id[match(colnames(counts), keep$column_name)]

samples <- analysis_samples[match(colnames(counts), analysis_samples$person_id), ]
stopifnot(identical(samples$person_id, colnames(counts)))

rm(counts_raw)   # free up memory
dim(counts)

## Part 5. Normalize and explore

### Sequencing depth

`colSums()` adds up each column: the total reads for each person, also called the **library size**. We divide by 1,000,000 to show it in millions.

In [ ]:
summary(colSums(counts) / 1e6)   # millions of reads per person

mean(rowSums(counts) == 0)       # share of genes with no reads in anyone

**🔍 Read the output:**

- Library sizes **vary** between people, which is why we normalize.
- Many genes have **no reads at all**: many of the 78,000 "genes" are inactive in these samples. They carry no information and are removed next.

### Filter and normalize with edgeR

| Step | Function | What it does |
|---|---|---|
| 1 | `DGEList()` | stores the counts, the gene table, and the sample table together in one object |
| 2 | `filterByExpr()` | finds genes with enough reads, in enough people, to analyse |
| 3 | `calcNormFactors()` | calculates a **normalization factor** for each person (the *TMM* method), adjusting for depth and composition |
| 4 | `cpm(log = TRUE)` | converts to **log₂ counts per million (log-CPM)**: normalized values on the log scale, for plots and PCA |

`dge[keep_genes, , keep.lib.sizes = FALSE]` keeps only the chosen genes (rows) and all people (columns).

In [ ]:
dge <- DGEList(counts = counts, genes = genes, samples = samples)

keep_genes <- filterByExpr(dge, group = samples$group)
table(keep_genes)

dge <- dge[keep_genes, , keep.lib.sizes = FALSE]
dge <- calcNormFactors(dge)

logcpm <- cpm(dge, log = TRUE)
dim(logcpm)

**🔍 Read the output:** `TRUE` is the number of genes we keep; about 29,000 in our test run, out of about 78,000.

### Why the log scale?

Compare the distribution of the raw counts with the log-CPM values. Each histogram uses the values for one person.

### Check the normalization

Normalization is a step you should always **look at**, not just run. Two checks:

1. **Normalization factors:** `calcNormFactors()` gives each person a TMM factor. Values close to **1** mean that person needed little composition correction; values far from 1 (say below 0.7 or above 1.4) are worth a closer look.
2. **Before and after:** boxplots of 30 people's values. **Before** (raw counts, log-transformed) the boxes sit at different heights, because people were sequenced to different depths. **After** (TMM-normalized log-CPM) the middle lines (medians) should line up.

`boxplot()` draws one box per column (person); `outline = FALSE` hides individual outlying genes; `xaxt = "n"` hides the x-axis labels (participant IDs).

In [ ]:
summary(dge$samples$norm.factors)   # TMM factors: usually between about 0.8 and 1.2

show <- 1:30   # the first 30 people
par(mfrow = c(1, 2))
boxplot(log2(dge$counts[, show] + 1), outline = FALSE, xaxt = "n",
        main = "Before: log2(raw counts + 1)", xlab = "30 people", ylab = "log2 scale")
boxplot(logcpm[, show], outline = FALSE, xaxt = "n",
        main = "After: TMM-normalized log-CPM", xlab = "30 people", ylab = "log2 CPM")
par(mfrow = c(1, 1))

**🔍 Read the output:** the factors cluster around 1. In the plots, the medians move from **uneven** (before) to roughly **level** (after), as they did in our test run. If one person's box still sat far from the rest after normalization, that sample could have a quality problem: check its `rqs` and position in the PCA.

`+ 1` before `log2()` avoids taking the log of 0 (which is minus infinity).

In [ ]:
par(mfrow = c(1, 2))   # two plots side by side
hist(counts[keep_genes, 1], breaks = 100, main = "Raw counts", xlab = "reads")
hist(logcpm[, 1],           breaks = 50,  main = "log-CPM",    xlab = "log2 counts per million")
par(mfrow = c(1, 1))

**🔍 Read the output:** raw counts are squashed against zero with a long tail: a few genes have enormous counts. On the log scale, the values are spread out and much easier to work with.

### PCA: the big picture

Each person has ~15,000 expression values. **Principal component analysis (PCA)** summarises them into a few new variables, the **principal components (PCs)**:

- **PC1** is the direction in which people differ the **most**; **PC2** the next most, and so on
- People with similar expression profiles appear **close together** on a PC1-vs-PC2 plot
- Each PC explains some **percentage of the variation**

We use the 1,000 **most variable** genes, as is common. `prcomp()` runs the PCA; `t()` **transposes** the matrix (flips rows and columns), because `prcomp()` wants one row per person.

In [ ]:
gene_variance <- apply(logcpm, 1, var)            # variance of each gene (row)
top_genes     <- order(gene_variance, decreasing = TRUE)[1:1000]

pca <- prcomp(t(logcpm[top_genes, ]))
pct <- round(100 * pca$sdev^2 / sum(pca$sdev^2), 1)  # % variation per PC

pca_df <- tibble(
  PC1          = pca$x[, 1],
  PC2          = pca$x[, 2],
  group        = samples$group,
  sex_at_birth = samples$sex_at_birth,
  age          = samples$age,
  rqs          = samples$rqs
)

ggplot(pca_df, aes(PC1, PC2, colour = group)) +
  geom_point(alpha = 0.6) +
  labs(x = str_glue("PC1 ({pct[1]}%)"), y = str_glue("PC2 ({pct[2]}%)"),
       colour = NULL, title = "PCA of gene expression")

**🔍 Read the output:** do cases and controls form separate clusters? Usually they **don't**: diabetes changes some genes a little, but it isn't the main source of variation. The big patterns in RNA-seq data often come from **technical** factors (how and when samples were processed, also called **batch effects**) or from **sex** and **cell composition**.

In our test run, PC1 split everyone into **two clear clusters** that had nothing to do with diabetes: almost certainly **sex** (genes on the Y chromosome, and *XIST*). The exercise below checks this.

That's why we don't stop at PCA: Part 6 tests each gene individually.

### ✍️ Your turn

1. Colour the PCA plot by **sex at birth** instead. Do the groups separate more than they did by diabetes? Why might that be?
2. Now colour it by **RNA quality** (`rqs`). Does PC1 or PC2 follow quality?

<details>
<summary><b>Show answer</b></summary>

```r
ggplot(pca_df, aes(PC1, PC2, colour = sex_at_birth)) +
  geom_point(alpha = 0.6)
```

Some genes are on the **X and Y chromosomes**, so their expression differs strongly by sex (for example, *XIST*, and genes on the Y chromosome that are only expressed by people with a Y chromosome). If these genes are among the 1,000 most variable, sex can show up clearly in a PC. If not, you may see little separation in PC1 and PC2: try `PC3` or `PC4` (`pca$x[, 3]`).

```r
ggplot(pca_df, aes(PC1, PC2, colour = rqs)) +
  geom_point(alpha = 0.6)
```

`rqs` is a number, so ggplot uses a colour **scale**. If the colours change steadily along PC1 or PC2, RNA quality is one of the main sources of variation: a **technical** effect, which is why it's in our model.

</details>

In [ ]:
# Your code here

## Part 6. Differential expression

We'll fit, for **every gene**, a linear model:

> expression ~ group + age + sex at birth + RNA quality

and test the **group** term (type 2 diabetes vs control). We use **limma**, which does this for all genes at once and borrows information across genes to make each test more reliable.

| Step | Function | What it does |
|---|---|---|
| 1 | `model.matrix()` | builds the **design matrix**: the model's variables as columns of numbers (like `glm()` does behind the scenes) |
| 2 | `voom()` | turns the counts into log-CPM values **with a weight** for each value, because low counts are less precise than high counts |
| 3 | `lmFit()` | fits the linear model to every gene |
| 4 | `eBayes()` | calculates the t-statistics and p-values, sharing information across genes |
| 5 | `topTable()` | returns a results table for one term (`groupcase`), sorted by p-value |

In [ ]:
design <- model.matrix(~ group + age + sex_at_birth + rqs, data = samples)
colnames(design)

v   <- voom(dge, design, plot = TRUE)   # plot = TRUE draws the mean-variance trend
fit <- eBayes(lmFit(v, design))

results <- topTable(fit, coef = "groupcase", number = Inf)

results %>%
  select(gene = Description, logFC, AveExpr, P.Value, adj.P.Val) %>%
  head(20)

**🔍 Read the output:** first, the **voom plot**. Each dot is a gene: its average expression (x) against how much it varies between people (y). The red line is the **trend**: genes with **low** counts are noisier, so voom gives their values **less weight** in the model. A smooth, downward trend is what you want (our test run's curve fell smoothly, then rose slightly for the very highest-count genes, which is normal). A sharp rise at the left means too many very-low-count genes survived filtering.

Then the results table:

| Column | Means |
|---|---|
| `gene` | the gene symbol |
| `logFC` | log₂ fold change, **type 2 diabetes vs control**. Positive = higher in people with diabetes; 1 = twice as high |
| `AveExpr` | the gene's average log-CPM: how active it is overall |
| `P.Value` | the p-value for this gene alone |
| `adj.P.Val` | the **adjusted p-value** (FDR), which accounts for testing ~15,000 genes. **Use this one.** |

`colnames(design)` shows the model's terms: `groupcase` is type 2 diabetes compared with the reference (*control*); `sex_at_birthMale` is Male compared with Female; `rqs` is RNA quality.

### How many genes are significant?

In [ ]:
results %>%
  mutate(direction = if_else(logFC > 0, "higher with diabetes", "lower with diabetes")) %>%
  filter(adj.P.Val < 0.05) %>%
  count(direction)

### What we found (test run)

With 300 cases and 300 controls, a few dozen genes had adjusted p < 0.05. Some patterns worth discussing:

- Several genes **higher** in diabetes (*MMP8*, *LTF*, *CRISP3*, *HP*, *ANXA3*) are made mostly by **neutrophils**, a kind of white blood cell. This may reflect a change in the **mix of cells** in the blood (more neutrophils, a sign of inflammation), not each cell changing its activity.
- ***ABCG1*** (cholesterol transport) was **lower** in diabetes. *ABCG1* has been linked to type 2 diabetes in earlier blood studies.
- The top gene, ***NRCAM***, has a large difference (logFC ≈ −0.7: about 40% lower) but is expressed at a **low** level overall (`AveExpr` below 0).

**💬 Discuss:** how could you tell a **cell-mix** effect from a real change in gene activity? (Ideas: adjust for each person's neutrophil count from their lab results in the `measurement` table; or estimate the cell mix from the expression data itself, called **deconvolution**.)

### Volcano plot

A **volcano plot** shows every gene at once:

- **x-axis:** the size of the difference (logFC). Further from 0 = bigger difference.
- **y-axis:** −log₁₀(p-value). Higher = stronger evidence. A p-value of 0.001 is 3; 0.000001 is 6.
- Genes **high and far to the left or right** are both large **and** convincing differences.

In [ ]:
ggplot(results, aes(logFC, -log10(P.Value), colour = adj.P.Val < 0.05)) +
  geom_point(size = 0.8, alpha = 0.6) +
  scale_colour_manual(values = c(`FALSE` = "grey60", `TRUE` = "firebrick"),
                      name = "Adjusted p < 0.05") +
  geom_vline(xintercept = 0, linetype = "dashed") +
  labs(x = "log2 fold change (type 2 diabetes vs control)", y = "-log10(p-value)",
       title = "Differential expression: type 2 diabetes")

### Look at the top gene

A boxplot of one gene's log-CPM values in each group. The box shows the middle half of the people; the line is the median.

In [ ]:
top_gene <- results$Name[1]

tibble(logCPM = logcpm[top_gene, ], group = samples$group) %>%
  ggplot(aes(group, logCPM)) +
  geom_boxplot() +
  labs(title = results$Description[1], y = "log2 counts per million")

**🔍 Read the output:** the groups overlap a lot, even for the most significant gene. Differences in expression between groups of people are usually **small shifts in the average**, not on/off switches. With hundreds of people, small shifts can still be clearly significant.

### Save the results

The results table has one row per **gene** (no participant data), so it's safe to save.

In [ ]:
save_data(results, "rnaseq_t2d_results.csv")

### Optional: which biological processes?

A list of genes is hard to interpret one by one. **Gene set enrichment** asks: are the genes that changed **concentrated** in particular biological processes, more than you'd expect by chance? We use limma's `goana()`, which tests every **Gene Ontology (GO)** term, a standard catalogue of processes (e.g. *neutrophil degranulation*), functions, and cell parts.

| Step | What it does |
|---|---|
| install `org.Hs.eg.db` and `GO.db` | Bioconductor **annotation** packages: gene IDs and GO terms (data only, no compiling). First time: a few minutes |
| `AnnotationDbi::mapIds()` | converts our Ensembl IDs to **Entrez** gene IDs, which `goana()` needs. We write `package::function()` instead of `library()`, because loading these packages would replace dplyr's `select()` |
| `goana(fit, coef = "groupcase", geneid = …)` | for each GO term, tests whether its genes are more often **up** or **down** in diabetes than other genes |
| `topGO(…, ontology = "BP")` | the top **biological processes** |

The comparison set (the **universe**) is all the genes we tested, not the whole genome: important, because only genes active in blood could have shown up.

In [ ]:
for (pkg in c("org.Hs.eg.db", "GO.db")) {
  if (!requireNamespace(pkg, quietly = TRUE)) BiocManager::install(pkg, lib = user_lib, update = FALSE, ask = FALSE)
}

entrez <- AnnotationDbi::mapIds(org.Hs.eg.db::org.Hs.eg.db, keys = str_remove(rownames(fit), "\\..*"),
                                keytype = "ENSEMBL", column = "ENTREZID", multiVals = "first")
mean(!is.na(entrez))   # share of our genes with an Entrez ID

go <- goana(fit, coef = "groupcase", geneid = entrez, FDR = 0.05)
topGO(go, ontology = "BP", number = 15)

**🔍 Read the output:**

| Column | Means |
|---|---|
| `Term` | the biological process |
| `N` | how many of our tested genes belong to it |
| `Up`, `Down` | how many of those are significantly **up** or **down** in diabetes |
| `P.Up`, `P.Down` | p-values: is that more than expected by chance? |

Look for themes, not single terms: GO terms overlap a lot, so the same genes can make several related terms significant. In our test run, almost every top term was **up** in diabetes and about the **immune system**: *immune system process* (p ≈ 7 × 10⁻⁶), *defense response*, *tumor necrosis factor production*, *phagocytosis*. That supports the neutrophil and inflammation pattern in the gene list.

Two cautions: each term rested on **few** genes (e.g. 14 of the 2,455 genes in "immune system process"), and with thousands of terms tested, p-values around 10⁻⁵ are **modest**. Treat enrichment as a supporting theme, not a finding on its own.

Two messages you may see: *'select()' returned 1:many mapping between keys and columns* is harmless (a few IDs match more than one gene; `multiVals = "first"` keeps the first). *No DE genes* means nothing passed the FDR threshold, so there's nothing to test for enrichment.

### ✍️ Your turn

1. The gene ***TXNIP*** is known to respond to blood sugar. Find it in `results`. Is it higher or lower in people with diabetes? Is it significant?
2. Run the model **without** `rqs`. How many genes are significant now? What does that tell you about RNA quality?

<details>
<summary><b>Show answer</b></summary>

1.
```r
results %>% filter(Description == "TXNIP") %>%
  select(gene = Description, logFC, adj.P.Val)
```
A positive `logFC` means higher in people with diabetes. It's significant if `adj.P.Val` is below 0.05.

2.
```r
design_no_rqs <- model.matrix(~ group + age + sex_at_birth, data = samples)
fit_no_rqs    <- eBayes(lmFit(voom(dge, design_no_rqs), design_no_rqs))
topTable(fit_no_rqs, coef = "groupcase", number = Inf) %>%
  summarise(significant = sum(adj.P.Val < 0.05))
```
If the number changes a lot, RNA quality differs between the groups, or adds noise that adjustment removes. Either way, technical factors can change your findings, so check them.

</details>

In [ ]:
# Your code here

### 🧩 Optional challenge: is it the cell mix? (about $1)

Are the neutrophil genes higher in diabetes because each neutrophil is more active, or just because people with diabetes have **more neutrophils** in their blood (a sign of inflammation)? If it's the cell mix, adding each person's **neutrophil count** to the model should shrink those genes' differences.

Neutrophil counts are in the `measurement` table, as **LOINC** code `751-8` (*Neutrophils [#/volume] in Blood by Automated count*). We average each person's results, then refit the model with `neutrophils` as one more covariate, for the people who have a count.

💰 The query scans the large `measurement` table: about $1. Skip this cell if you're short on time or budget.

In [ ]:
neutrophil_sql <- str_glue("
SELECT m.person_id, AVG(m.value_as_number) AS neutrophils
FROM `{CURRENT_DATASET}.measurement` AS m
JOIN `{CURRENT_DATASET}.concept` AS c ON m.measurement_concept_id = c.concept_id
WHERE c.vocabulary_id = 'LOINC' AND c.concept_code = '751-8'
  AND m.value_as_number BETWEEN 0 AND 50
  AND m.person_id IN ({str_c(samples$person_id, collapse = ', ')})
GROUP BY m.person_id")

neutrophils <- run_query(neutrophil_sql) %>% mutate(person_id = as.character(person_id))

with_neut <- samples %>% left_join(neutrophils, by = "person_id")
has_value <- !is.na(with_neut$neutrophils)
sum(has_value)   # people with a neutrophil count

design_neut <- model.matrix(~ group + age + sex_at_birth + rqs + neutrophils, data = with_neut[has_value, ])
fit_neut    <- eBayes(lmFit(voom(dge[, has_value], design_neut), design_neut))

topTable(fit_neut, coef = "groupcase", number = Inf) %>%
  filter(Description %in% c("MMP8", "LTF", "CRISP3", "HP", "ANXA3")) %>%
  select(gene = Description, logFC, adj.P.Val)

# For comparison: the same genes in the main model (without neutrophil counts)
results %>%
  filter(Description %in% c("MMP8", "LTF", "CRISP3", "HP", "ANXA3")) %>%
  select(gene = Description, logFC, adj.P.Val)

**🔍 Read the output:** the number of people with a neutrophil count, then the five neutrophil genes **with** the count in the model, then **without** (the main results). If their logFC shrinks toward 0 with the count in the model, much of the "diabetes effect" was a difference in **cell mix**, not in what each cell does.

Things to discuss: only some of our 600 people have a count, so the model is smaller (less power); the counts come from **any** date, not the day of the RNA sample; and units are usually 10³ cells per µL, but not always (the `BETWEEN 0 AND 50` line drops obvious unit mix-ups).

## Part 7. Back to *TCF7L2* and rs7903146

In 4.1 to 4.4, the **T allele** of rs7903146, in the gene *TCF7L2*, was linked to type 2 diabetes. Variants that change disease risk often work by changing **how much** of a gene is made. A variant linked to a gene's expression is an **eQTL** (*expression quantitative trait locus*).

Two questions:

1. Is ***TCF7L2*** expressed differently in people with type 2 diabetes?
2. Does the **T allele** change *TCF7L2* expression in blood?

### *TCF7L2* in our results

In [ ]:
results %>%
  filter(Description == "TCF7L2") %>%
  select(gene = Description, logFC, AveExpr, P.Value, adj.P.Val)

**🔍 Read the output:** *TCF7L2*'s logFC and adjusted p-value. (If the table is empty, the gene was removed by `filterByExpr()`: too few reads in blood.) A gene can matter for a disease **without** being expressed differently in blood: *TCF7L2* acts mainly in the **pancreas**, gut, and fat tissue. In our test run, *TCF7L2* was clearly expressed in blood but **not** different between cases and controls (adjusted p ≈ 0.95).

### Genotype vs expression: our own eQTL test

We get rs7903146 genotypes (copies of **T**) for our 600 people with PLINK, as in 4.1 and 4.4, and test:

> *TCF7L2* expression (log-CPM) ~ copies of T + age + sex + RNA quality

with `lm()`, a **linear** regression, because expression is a number, not yes/no. Most people with RNA-seq also have a genome; anyone without one drops out (`NA`).

In [ ]:
pgen_dir    <- file.path(aou_data_dir("v9"), "wgs", "short_read", "snpindel", "acaf_threshold", "pgen")
GENO_DIR    <- file.path(DATA_DIR, "genomics")
dir.create(GENO_DIR, showWarnings = FALSE)
keep_file   <- file.path(GENO_DIR, "keep_rnaseq.txt")
allele_file <- file.path(GENO_DIR, "count_allele.txt")
lead_prefix <- file.path(GENO_DIR, "rs7903146_rnaseq")

write_lines(samples$person_id, keep_file)
write_lines("chr10:112998590:C\tT", allele_file)   # count the T allele (4.1 Part 6)

plink_output <- system(str_glue(
  "plink2 --pfile {shQuote(file.path(pgen_dir, 'acaf_threshold.chr10'))} ",
  "--keep {shQuote(keep_file)} --chr 10 --from-bp 112998590 --to-bp 112998590 ",
  "--export A --export-allele {shQuote(allele_file)} --out {shQuote(lead_prefix)}"
), intern = TRUE)

genotypes <- read_tsv(paste0(lead_prefix, ".raw"), col_types = cols(.default = "c"))
geno_col  <- names(genotypes)[ncol(genotypes)]

tcf7l2_row <- which(dge$genes$Description == "TCF7L2")[1]

eqtl_data <- samples %>%
  mutate(tcf7l2 = logcpm[tcf7l2_row, ]) %>%
  left_join(genotypes %>% transmute(person_id = IID, copies_T = as.numeric(.data[[geno_col]])),
            by = "person_id")

eqtl_data %>% count(copies_T) %>% hide_small_counts()   # people with 0, 1, 2 copies (NA = no genome)

lm(tcf7l2 ~ copies_T + age + sex_at_birth + rqs, data = eqtl_data) %>%
  broom::tidy(conf.int = TRUE) %>%
  filter(term == "copies_T")

**🔍 Read the output:**

- The genotype counts: how many of our people have 0, 1, or 2 copies of T.
- `estimate`: the change in *TCF7L2* log-CPM **per copy of T**. On the log₂ scale, 0.1 means about 7% more expression per copy.
- `p.value`: with only 600 people, a small effect may not be significant. In our test run there was **no** sign of an effect (estimate ≈ −0.01, p ≈ 0.6), and the boxes below were almost identical. Let's check *All of Us*'s own eQTL results, from all ~9,000 people.

Visualise it: one box per genotype.

In [ ]:
eqtl_data %>%
  filter(!is.na(copies_T)) %>%
  ggplot(aes(factor(copies_T), tcf7l2)) +
  geom_boxplot() +
  labs(x = "Copies of the T allele (rs7903146)", y = "TCF7L2 expression (log2 CPM)",
       title = "TCF7L2 expression by rs7903146 genotype, whole blood")

### The *All of Us* eQTL results

*All of Us* has tested every gene against nearby variants, in all participants with RNA-seq (**COMB**, combined) and in each ancestry group. Two kinds of files:

| Folder | Has | Size |
|---|---|---|
| `eqtl/ciseqtl/` | **one row per gene**: its strongest nearby variant | small |
| `eqtl/cis_eqtl_nominal_stats/` | **every** gene–variant pair tested, one file per chromosome (`.parquet`) | about 1 GB for chromosome 10 |

First, *TCF7L2*'s strongest eQTL variant. These are summary statistics: no individual data.

| Column | Means |
|---|---|
| `phenotype_id` | the gene's Ensembl ID |
| `variant_id` | the variant, as `chr10:112998590_C_T` |
| `slope` | the change in expression per copy of the **ALT** allele (in normalized units) |
| `pval_nominal`, `pval_beta`, `qval` | the p-value; a p-value corrected for testing many variants per gene; and the false discovery rate across genes |

In [ ]:
TCF7L2_ID <- "ENSG00000148737"   # TCF7L2's Ensembl gene ID

eqtl_genes <- read_tsv(file.path(RNASEQ_DIR, "eqtl", "ciseqtl", "COMB.cis_qtl.txt.gz"), show_col_types = FALSE)

eqtl_genes %>%
  filter(str_starts(phenotype_id, TCF7L2_ID)) %>%
  select(phenotype_id, variant_id, af, slope, pval_nominal, pval_beta, qval)

**🔍 Read the output:** *TCF7L2*'s strongest eQTL in blood. Is it rs7903146 (`chr10:112998590_C_T`)? How small is `qval`? A small `qval` (below 0.05) means *TCF7L2* has **some** eQTL in blood, though not necessarily rs7903146.

In our test run, the strongest was a **different** variant, `chr10:112951996_C_T`, near the start of the gene (about 47,000 letters from rs7903146), with a strong effect (`slope` ≈ −0.25) and p-values shown as **0**. A p-value of 0 doesn't mean "impossible": it's smaller than the smallest number the file can store (below about 10⁻³⁰⁸).

### rs7903146 in every gene–variant test

The nominal results file is too big to load whole. R's `arrow` package (for `.parquet` files) isn't installed in this app, but Python's `pyarrow` is, so we write a tiny Python script that reads **only the rows for rs7903146** and saves them as a small text file, then read that into R.

In [ ]:
nominal_file <- file.path(RNASEQ_DIR, "eqtl", "cis_eqtl_nominal_stats", "COMB", "COMB.cis_qtl_pairs.chr10.parquet")
lookup_file  <- file.path(tempdir(), "rs7903146_eqtl.tsv")
script_file  <- file.path(tempdir(), "lookup.py")

write_lines(c(
  "import pyarrow.parquet as pq",
  str_glue("table = pq.read_table('{nominal_file}', filters=[('variant_id', '==', 'chr10:112998590_C_T')])"),
  str_glue("table.to_pandas().to_csv('{lookup_file}', sep='\\t', index=False)")
), script_file)

python_output <- system(str_glue("python {shQuote(script_file)}"), intern = TRUE)   # takes a minute or so

rs7903146_eqtl <- read_tsv(lookup_file, show_col_types = FALSE)

# Gene symbols, from the count file's gene table (IDs without the version number)
gene_names <- genes %>% transmute(gene_id = str_remove(Name, "\\..*"), gene = Description)

rs7903146_eqtl %>%
  mutate(gene_id = str_remove(phenotype_id, "\\..*")) %>%
  left_join(gene_names, by = "gene_id") %>%
  arrange(pval_nominal) %>%
  select(gene, phenotype_id, start_distance, slope, pval_nominal)

**🔍 Read the output:** every gene near rs7903146 that was tested against it, sorted by p-value. Find *TCF7L2*'s row; `slope` is per copy of **T** (the ALT allele). `start_distance` is how far the variant is from the gene's start.

In our test run, *TCF7L2* was the top row, but with a **tiny** effect (`slope` ≈ 0.02, compared with −0.25 for the gene's strongest eQTL) and only a modest p-value (≈ 0.004) despite ~9,000 people. So in **blood**, rs7903146 barely changes *TCF7L2* expression, and our 600-person test had no chance of seeing it.

**💬 Discuss:** rs7903146 is one of the strongest genetic risk factors for type 2 diabetes. If it has **little or no** eQTL effect on *TCF7L2* in **blood**, what might that mean? (Hint: genes are regulated differently in each tissue. Studies of **pancreatic islets**, where insulin is made, have linked the T allele to *TCF7L2* regulation there. A blood sample can't show what happens in the pancreas: an important limit of blood-based omics.)

👩‍🏫 **Instructor tip:** gene IDs in the eQTL files have a version number (`ENSG00000148737.18`); `str_starts()` matches the ID without worrying about the version.

### ✍️ Your turn

Which of your **significant** differentially expressed genes (Part 6) have a strong eQTL in blood?

<details>
<summary><b>Show answer</b></summary>

```r
significant <- results %>%
  filter(adj.P.Val < 0.05) %>%
  mutate(gene_id = str_remove(Name, "\\..*")) %>%   # remove the version number
  select(gene_id, gene = Description, logFC, adj.P.Val)

eqtl_genes %>%
  mutate(gene_id = str_remove(phenotype_id, "\\..*")) %>%
  inner_join(significant, by = "gene_id") %>%
  filter(qval < 0.05) %>%
  select(gene, logFC, adj.P.Val, variant_id, slope, qval) %>%
  arrange(qval)
```

A gene that's both linked to diabetes **and** controlled by a variant is a lead for further study: is that variant also linked to diabetes in a GWAS (4.2) or in All by All (4.4)?

</details>

In [ ]:
# Your code here

## Discuss

1. **Cause or effect?** If a gene is higher in people with diabetes, did it **cause** the diabetes, or is it a **result** of diabetes (or of high blood sugar)?
2. **Medications:** many people with diabetes take drugs like metformin. Could a drug, rather than the disease, explain some differences? How could you check?
3. **Other confounders:** we adjusted for age and sex. What else differs between the groups and affects gene expression? (Think of BMI, other conditions, and the mix of cell types in the sample.)
4. **Definition of diabetes:** our cohort means "ever had a type 2 diabetes diagnosis in the health records". It doesn't say **when**, or whether the blood sample was taken before or after the diagnosis. Why might that matter?
5. **Tissue:** what can blood tell us about a disease of the pancreas, liver, and muscle? What can't it?

## ⚠️ Common errors

| Message | Means |
|---|---|
| *Can't find the All of Us v9 data folder* | The Controlled Tier data collection isn't added to this workspace |
| *cannot open file '../aou_setup.R'* | The notebook isn't in `notebooks/omics/` |
| *there is no package called 'edgeR'* | Rerun the Setup cell and wait for the install to finish |
| *cannot allocate vector of size …* | The app ran out of memory: lower `N_PER_GROUP`, or use an app with more memory |
| *stopifnot(…) is not TRUE* | The sample table and the count columns don't match. Rerun Part 4 from the start. |
| `tcf7l2_row` is `NA` | *TCF7L2* was filtered out by `filterByExpr()`. Use `dge$genes` before filtering, or lower the filter |
| *ModuleNotFoundError: No module named 'pyarrow'* | The Python lookup needs `pyarrow`, which is in the *All of Us* app. Check `system("python -c 'import pyarrow'")` |
| *object 'samples' not found* | Run the cells in order; Part 4 creates `samples` |

## ✅ Key takeaways

- RNA-seq counts how many reads come from each gene in each person
- Use **counts** for comparing groups; **normalize** for sequencing depth, and work on the **log scale**
- **PCA** shows the main patterns in the data, which are often technical, not the disease
- **Differential expression** fits one model per gene; read the **logFC** and the **adjusted p-value**
- Adjust for **technical** factors such as RNA quality, as well as age and sex
- Results are **associations**: think about confounders, medications, timing, and **tissue**
- An **eQTL** links a variant to a gene's expression: the bridge between DNA (4.1–4.4) and RNA

**Going further:**

- Increase `N_PER_GROUP` (or use everyone) for more power. Use an app with more memory, and expect the loading step to take longer.
- Add more covariates from the Controlled Tier dataset, such as BMI, to the model
- Repeat the eQTL lookup with an ancestry-specific file (e.g. `EUR.cis_qtl.txt.gz`, `AFR.cis_qtl.txt.gz`): are the results the same?

**Next:** **4.6 proteomics**: from RNA to **proteins**, measured in blood plasma.

**Before you share or commit this notebook:** clear all outputs (**Edit → Clear Outputs of All Cells**).